In [1]:
from glob import glob
import os
from os.path import join

import h5py
from matplotlib import pyplot as plt
import numpy as np
from o2_utils.selectors import find_files_from_pattern
from PIL import Image
from sklearn.cluster import KMeans
from tqdm.notebook import tqdm
import videochef as vc

from multicamera_airflow_pipeline.jonah_241112.skeletons.defaults import kpt_dict

In [2]:
preds_2d_path = "/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions"
# mice_to_use = ["J08101", "J08102"]  # need to compress these videos, nb might offset frames by 1 wrt already detected kpts :/ 
mice_to_use = ["J05102", "J05105"]
vid_dir_by_mouse = {
    "J08101": "/n/groups/datta/Jonah/20241029_PBN_Tac1_stim/raw_data/J08101",
    "J08102": "/n/groups/datta/Jonah/20241029_PBN_Tac1_stim/raw_data/J08102",
    "J05102": "/n/groups/datta/Jonah/20240520_vlPAG_Tac1_photom/20240520_6cam/data/J05102",
    "J05105": "/n/groups/datta/Jonah/20240520_vlPAG_Tac1_photom/20240520_6cam/data/J05105",
}
save_dir = "/n/groups/datta/Jonah/20240820_airflow_pipeline/more_training_data"

## Extract mid-conf forepaw frames for side cameras
* The video reading is sped up quite a bit by having multiple CPUs (I'm using 8 cpus and getting a ~4x speedup relative to 1 cpu)

In [5]:
cameras = ["bottom", "side1", "side2", "side3", "side4"]
keypoint_names = list(kpt_dict.keys())
forepaw_names = ["left_fore_paw", "right_fore_paw"]
downsample = 20
n_frames_for_clustering = 1000  # frames per vid
n_clusters = 6
n_frames_per_cluster = 20

for mouse in mice_to_use:
    sessions = glob(join(preds_2d_path, f"*{mouse}*"))
    
    for session in sessions:

        # Setup for this session
        base_name = os.path.basename(session)
        save_path = join(save_dir, "midconf_images", base_name)
        os.makedirs(save_path, exist_ok=True)
        files = find_files_from_pattern(session, "*.h5", error_behav="pass")
        if files is None:
            continue

        for camera in cameras:
        # for camera in ["side1"]:

            # Check if done already
            saved_imgs = find_files_from_pattern(save_path, f"{base_name}.{camera}.*.png", error_behav="pass")
            if saved_imgs is not None and len(saved_imgs) > 0:
                print(f"Already saved images for {base_name} {camera}")
                continue

            # Find file for this session / camera
            file = find_files_from_pattern(session, f"*.{camera}*.h5", error_behav="pass")
            if file is None: 
                continue
            print(file)

            # Load data
            with h5py.File(file, "r") as h5f:
                keypoint_coords = np.array(h5f["keypoint_coords"])
                keypoint_conf = np.array(h5f["keypoint_conf"])
                detection_conf = np.array(h5f["detection_conf"])
                detection_coords = np.array(h5f["detection_coords"])
            
            # Find relevant frames
            midconf_frame_ixs = []
            for fp in forepaw_names:
                fp_confs = keypoint_conf[:, :, keypoint_names.index(fp)]
                midconf_frame_ixs.append(np.where((fp_confs > 0.4) & ((fp_confs < 0.65)))[0])
            midconf_frame_ixs = np.unique(np.concatenate(midconf_frame_ixs))
            midconf_frame_ixs = np.random.choice(midconf_frame_ixs, n_frames_for_clustering, replace=False)
            midconf_frame_ixs = np.sort(midconf_frame_ixs)

            # Load the frames and save downsampled versions
            raw_vid = glob(join(vid_dir_by_mouse[mouse], base_name, f"{base_name}*{camera}*.mp4"))[0]
            with vc.io.VideoReader(raw_vid, frame_ixs=midconf_frame_ixs) as vr:
                frame0 = next(vr)
            frame_shape = frame0.shape
            frames = np.zeros((n_frames_for_clustering, *frame_shape[:2]))
            iFrame = 0
            # with vc.io.VideoReader(raw_vid, frame_ixs=midconf_frame_ixs, reporter_val=0) as vr:
            with vc.io.VideoReader(raw_vid, frame_ixs=midconf_frame_ixs) as vr:
                for frame_ix, frame in tqdm(zip(midconf_frame_ixs, vr), total=n_frames_for_clustering):
                    # save the frame from np array
                    # Image.fromarray(frame).save(join(save_path, f"{base_name}.{camera}.frame_{frame_ix}.png"))

                    # Store frame
                    frames[iFrame] = frame[:, :, 0]
                    iFrame += 1
            
            # Perform kmeans clustering on frames from this camera
            frames = np.array(frames)  # (n_frames, h, w)
            frames_downsampled = frames[:, ::downsample, ::downsample]  # (n_frames, h, w)
            X = frames_downsampled.reshape(-1, frames_downsampled.shape[1] * frames_downsampled.shape[2])  # (n_frames, h*w)
            model = KMeans(n_clusters=n_clusters, random_state=0)
            model.fit(X)
            labels = model.predict(X)

            # Sample frames from each cluster
            total_frames_saved = 0
            saved_frame_ixs = []
            for lab in np.unique(labels):
                choices = np.where(labels == lab)[0]
                chosen_frame_ixs_to_save = np.random.choice(choices, np.min([n_frames_per_cluster, len(choices)]), replace=False)
                for chosen_frame_ix in chosen_frame_ixs_to_save:
                    frame = frames[chosen_frame_ix]
                    Image.fromarray(frame).convert('RGB').save(join(save_path, f"{base_name}.{camera}.frame_{midconf_frame_ixs[chosen_frame_ix]}.png"))
                    saved_frame_ixs.append(chosen_frame_ix)
                    total_frames_saved += 1

            # Save out any more random frames to make up the total
            # extra_frames_to_save = (n_frames_per_cluster * n_clusters) - total_frames_saved
            # for i in np.arange(extra_frames_to_save):
            #     chosen_frame_ix = np.random.choice(np.where(~np.isin(np.arange(n_frames_for_clustering), saved_frame_ixs))[0])
            #     frame = frames[chosen_frame_ix]
            #     Image.fromarray(frame).convert('RGB').save(join(save_path, f"{base_name}.{camera}.frame_{chosen_frame_ix}.png"))


/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240626_J05102/20240626_J05102.bottom.0.h5


  0%|          | 0/1000 [00:00<?, ?it/s]

/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240626_J05102/20240626_J05102.side1.0.h5


  0%|          | 0/1000 [00:00<?, ?it/s]

/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240626_J05102/20240626_J05102.side2.0.h5


...

  0%|          | 0/1000 [00:00<?, ?it/s]

/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240627_J05105/20240627_J05105.side2.0.h5


  0%|          | 0/1000 [00:00<?, ?it/s]

/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240627_J05105/20240627_J05105.side3.0.h5


  0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
# for label in np.unique(labels):
#     nrows = np.ceil(np.sqrt(np.sum(labels == label))).astype(int)
#     ncols = np.ceil(np.sum(labels == label) / nrows).astype(int)
#     plt.subplots(nrows, ncols, figsize=(ncols*3, nrows*3))
#     chosen_frame_ixs = np.where(labels == label)[0]
#     for i, ix in enumerate(chosen_frame_ixs):
#         plt.subplot(nrows, ncols, i+1)
#         plt.imshow(frames[ix], cmap="gray")
#         plt.axis("off")
#         plt.title(f"Frame {ix}")
#     plt.subplots_adjust(wspace=0, hspace=0)
